### LangChain RAG with chat history and streaming

#### Dependencies

We'll use the following packages:

In [32]:
%pip install --upgrade --quiet langchain langchain-community langchain-openai langchain_huggingface python-dotenv langgraph

Note: you may need to restart the kernel to use updated packages.


#### Load API keys into environment variables

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

#### Define LLM

In [1]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o', temperature=0)

### Define text embeddings

In [2]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

# from langchain_huggingface import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(model_name='intfloat/e5-large-v2')
# embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2') # 512 token limit, after which it will truncate the input


### Load document into local vector store 

- PDF documents are converted to Markdown using pymupdf4llm
- Markdown documents are chunked and linked to parent for Child-Parent retrieval

In [75]:
from langchain_core.vectorstores import InMemoryVectorStore
vector_store = InMemoryVectorStore(embeddings)

import pymupdf4llm

def pdf_to_md(path):
  return pymupdf4llm.to_markdown(path)

from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

import re

CHUNK_SIZE = 8191 * 4 # Maximum size of text (in characters) that can be processed by OpenAI text-embedding-ada-002 embedding model
CHUNK_OVERLAP = 200

def chunk_markdown(markdown, source):
    docs = MarkdownHeaderTextSplitter(
        headers_to_split_on = [ ('#', 'Header 1'), ('##', 'Header 2'), ('###', 'Header 3') ], 
        strip_headers=False
    ).split_text(markdown)

    # Char-level splits
    docs = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, 
        chunk_overlap=CHUNK_OVERLAP
    ).split_documents(docs)
  
    last_heading_1_id = None
    last_heading_2_id = None
    for idx in range(len(docs)):
        doc = docs[idx]
        doc.metadata['source'] = source
        headings_ids = []
        for heading in [doc.metadata[key] for key in sorted(doc.metadata) if key.startswith('Header')]:
            headings_ids.append(re.sub(r'[^a-zA-Z0-9]', '',''.join([token.title() for token in heading.split()])))
        doc.id = f'{source.replace("/",":")}:{":".join(headings_ids)}'
        doc.metadata['id'] = f'{source.replace("/",":")}:{":".join(headings_ids)}'

        if 'Header 1' in doc.metadata and 'Header 2' not in doc.metadata:
            last_heading_1_id = doc.metadata['id']
        if 'Header 2' in doc.metadata and 'Header 3' not in doc.metadata:
            last_heading_2_id = doc.metadata['id']
            doc.metadata['parent_id'] = last_heading_1_id
        if 'Header 3' in doc.metadata:
            doc.metadata['parent_id'] = last_heading_2_id
    return docs

def load(path, vector_store):
    if path.endswith('.pdf'):
        markdown = pdf_to_md(path)
    elif path.endswith('.md'):
        markdown = open(path).read()
    else:
        raise ValueError('Unsupported file format')
    docs = chunk_markdown(markdown, path)
    vector_store.add_documents(documents=docs)

load('general/2025HoaFees.md', vector_store)

retriever = vector_store.as_retriever()

[InMemoryVectorStore](https://python.langchain.com/api_reference/core/vectorstores/langchain_core.vectorstores.in_memory.InMemoryVectorStore.html)

### Define our question

In [76]:
question = 'list the hoa monthly fees by neighborhood, including the base assessment and any additional fees'

### Search the vector store

In [80]:
def retrieve_from_vector_store(question):
    retrieved_docs = vector_store.similarity_search(question) # k=4 by default

    # If the retrieved documents are sub-sections of a larger document, prepend the parent document content to the retrieved document content
    parent_ids = set([doc.metadata['parent_id'] for doc in retrieved_docs if 'parent_id' in doc.metadata])
    parent_chunks = dict([(doc.id, doc) for doc in vector_store.get_by_ids(list(parent_ids))])
    for doc in retrieved_docs:
        if 'parent_id' in doc.metadata and doc.metadata['parent_id'] in parent_chunks:
            doc.page_content = parent_chunks[doc.metadata['parent_id']].page_content + '\n' + doc.page_content

    return retrieved_docs

#### Define a simple RAG chain using LangGraph

In [81]:
from langchain import hub
from langchain_core.documents import Document
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict

# Define prompt for question-answering
prompt = hub.pull('rlm/rag-prompt')

# Define state for application
class State(TypedDict):
    question: str
    context: List[Document]
    answer: str

# Define application steps
def retrieve(state: State):
    return {'context': retrieve_from_vector_store(state['question'])}

def generate(state: State):
    docs_content = '\n\n'.join(doc.page_content for doc in state['context'])
    messages = prompt.invoke({'question': state['question'], 'context': docs_content})
    response = llm.invoke(messages)
    return {'answer': response.content}

# Compile application
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, 'retrieve')
graph = graph_builder.compile()

#### Submit the question to the LLM with the applicable vector store doc(s)

In [82]:
response = graph.invoke({'question': question})
print(response['answer'])

The HOA monthly fees by neighborhood, including the base assessment, are as follows:

- Coastal Villas: $736
- Live Oak Village: $745
- Egret Cove: $744
- The Aviary: $752.5
- The Preserve: $763.5
- Seaford Place: $761
- Summerplace Village: $757
- Fishery Bluff: $391
- Argent Cottages: $300
- Plymouth Cottages: $288.5
- Andover Cottages: $306
- Cypress Hollow: $299.5
- Argent II and Sun City West: $295

These amounts include the base assessment of $224.
